<a href="https://colab.research.google.com/github/JulietChinenyeDuru/KG-SentIF-Knowledge-Graph-Enhanced-Sentiment-Analysis-for-Nigerian-Social-Media-Text/blob/main/%20%20%20notebooks/google_play_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NaijaABSA:Google Play reviews

**What this notebook does**
1. Uses a fixed list of Nigerian apps
2. Downloads recent reviews (reviewer names are never saved)
3. Removes personal details, duplicates and very short or long reviews
4. Keeps reviews containing Nigerian Pidgin markers


## 1. Install the library

In [ ]:
!pip install -q google-play-scraper

In [ ]:
# I collect Google Play reviews from Nigerian apps in ten sectors and save each app to my Google Drive so nothing is lost

!pip install -q google-play-scraper
import os, time
import pandas as pd
from google_play_scraper import reviews, Sort
from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/NaijaABSA/raw_reviews"
os.makedirs(SAVE_DIR, exist_ok=True)

APPS = {
    "telecoms":      ["ng.mtn.nextgen", "net.one97.selfcare.globacom", "com.airtel.africa.selfcare"],
    "banking":       ["com.firstbank.firstmobile", "com.moniepoint.business", "com.moniepoint.personal",
                      "team.opay.pay", "com.kudabank.app", "com.transsnet.palmpay",
                      "com.gtbank.gtworldv1", "com.accessbank.nextgen", "com.ubanquity.redd.uba"],
    "electricity":   ["com.vatebrapay.fcmbenergy", "com.ionicframework.buypowerapp334891", "com.powerlight.ng"],
    "ecommerce":     ["com.jumia.android", "ng.jiji.app", "com.konga.androida"],
    "ride-hailing":  ["ee.mtakso.client", "sinet.startup.inDriver", "com.ubercab"],
    "food-delivery": ["com.chowdeck.com", "com.glovo"],
    "education":     ["com.ulesson"],
    "travel":        ["com.flyairpeace.app.airpeace", "com.gigm", "com.wakanow.android"],
    "government":    ["com.nimcmobile"],
    "pay-tv":        ["com.dstv.mydstv", "com.dstv.mygotv", "com.multichoice.gotv", "com.dstvmobile.android"],
}
MAX_PAGES = 150

for domain, ids in APPS.items():
    for app_id in ids:
        out = f"{SAVE_DIR}/{app_id}.csv"
        if os.path.exists(out):
            print(f"Already saved, skipping: {app_id}")
            continue
        rows, token = [], None
        for _ in range(MAX_PAGES):
            try:
                batch, token = reviews(app_id, lang="en", country="ng", sort=Sort.NEWEST,
                                       count=200, continuation_token=token)
            except Exception as e:
                print("Problem with", app_id, "-", e); break
            rows += [{"review_id": r["reviewId"], "app_id": app_id, "domain": domain,
                      "text": r["content"], "date": str(r["at"])} for r in batch]
            if not batch or token is None:
                break
            time.sleep(2)
        pd.DataFrame(rows, columns=["review_id", "app_id", "domain", "text", "date"]).to_csv(out, index=False)
        print(f"{domain:14s} {app_id:40s} {len(rows)} reviews saved")

Mounted at /content/drive
telecoms       ng.mtn.nextgen                           30000 reviews saved
telecoms       net.one97.selfcare.globacom              16385 reviews saved
telecoms       com.airtel.africa.selfcare               30000 reviews saved
banking        com.firstbank.firstmobile                30000 reviews saved
banking        com.moniepoint.business                  30000 reviews saved
banking        com.moniepoint.personal                  30000 reviews saved
banking        team.opay.pay                            30000 reviews saved
banking        com.kudabank.app                         30000 reviews saved
banking        com.transsnet.palmpay                    30000 reviews saved
banking        com.gtbank.gtworldv1                     30000 reviews saved
banking        com.accessbank.nextgen                   30000 reviews saved
banking        com.ubanquity.redd.uba                   14028 reviews saved
electricity    com.vatebrapay.fcmbenergy                10 rev

In [ ]:
# I download my Pidgin review pool from Google Drive to my computer

from google.colab import drive, files
drive.mount("/content/drive")
files.download("/content/drive/MyDrive/NaijaABSA/pidgin_pool.jsonl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# I convert my Pidgin review pool into a CSV file that opens in Excel or Google Sheets

import pandas as pd
from google.colab import drive, files

drive.mount("/content/drive")
df = pd.read_json("/content/drive/MyDrive/NaijaABSA/pidgin_pool.jsonl", lines=True)
df.to_csv("pidgin_pool.csv", index=False, encoding="utf-8-sig")
print(len(df), "reviews")
files.download("pidgin_pool.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2526 reviews


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# I remove non-Nigerian, foreign-language reviews from my Pidgin pool and save the final set of Nigerian Pidgin reviews

!pip install -q langdetect==1.0.9
import re
import pandas as pd
from langdetect import detect_langs, DetectorFactory
from google.colab import drive, files

DetectorFactory.seed = 0
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/NaijaABSA"

df = pd.read_json(f"{BASE}/pidgin_pool.jsonl", lines=True)
print("Pidgin pool before cleaning:", len(df))

STRONG = ["dey", "wey", "abeg", "no be", "wetin", "sabi", "oga", "wahala", "comot", "abi", "sha",
          "shey", "e don", "don dey", "na im", "nawa", "chop", "o jare", "biko", "make una", "e no",
          "una don", "una dey", "una go", "una no"]
strong_pattern = r"\b(?:" + "|".join(re.escape(m) for m in STRONG) + r")\b"
low = df["text"].str.lower()

def main_language(text):
    try:
        return detect_langs(text)[0].lang
    except Exception:
        return "unknown"

df["language_detected"] = df["text"].map(main_language)
FOREIGN = ["es", "pt", "it", "ca", "fr", "de", "ro", "gl"]
rule_1 = df["language_detected"].isin(FOREIGN) & (low.str.count(strong_pattern) == 0)

PIDGIN_WITHOUT_SHA = [m for m in STRONG if m != "sha"] + ["una"]
pattern_without_sha = r"\b(?:" + "|".join(re.escape(m) for m in PIDGIN_WITHOUT_SHA) + r")\b"
rule_2 = low.str.contains(r"ma\s*sha\.?\s*allah") & (low.str.count(pattern_without_sha) == 0)

CHECKED_BY_HAND = [
    "d11f1915-da52-4f3a-ad00-272a5a6956b6",  # Swahili
    "ce161e26-24da-4843-9580-40772bfe694a",  # Swahili
    "7628565a-62dd-47a1-98ed-6a12e44c53a8",  # Somali
    "8990f431-a33b-414b-9893-4e961605ab24",  # Estonian
    "0f0eb3a3-be18-41c1-a1ef-b632dadd798f",  # Malay
    "f973307f-190d-4b59-a9bd-839411f2740b",  # Hindi/Urdu
    "1e80e671-401a-4c15-8cd6-028175115d44",  # Urdu
    "d2c93fe6-e5d7-4c09-9aac-31d3a58208e3",  # Khasi
    "04d3398b-f4a0-4a5a-8c18-037db0058745",  # Croatian
]
rule_3 = df["review_id"].isin(CHECKED_BY_HAND)

removed = df[rule_1 | rule_2 | rule_3]
final = df[~(rule_1 | rule_2 | rule_3)].drop(columns="language_detected")

print("Removed as foreign-language:", len(removed))
print("Final Nigerian Pidgin reviews:", len(final))
print(final["domain"].value_counts())

final.to_json(f"{BASE}/nigerian_pidgin_reviews.jsonl", orient="records", lines=True)
removed.to_csv(f"{BASE}/removed_foreign_reviews.csv", index=False)
final.to_csv("nigerian_pidgin_reviews.csv", index=False, encoding="utf-8-sig")
files.download("nigerian_pidgin_reviews.csv")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pidgin pool before cleaning: 2526
Removed as foreign-language: 175
Final Nigerian Pidgin reviews: 2351
domain
banking          1147
telecoms          562
government        191
ecommerce         162
pay-tv            123
electricity        49
food-delivery      49
ride-hailing       36
travel             24
education           8
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# I install the krippendorff package and print its version, so I can report it in my paper

!pip install -q krippendorff
from importlib.metadata import version
print(version("krippendorff"))

0.9.0


In [5]:
# I combine my two agreement sheets (A and B) into one file of review IDs and both sets of labels, without the review text

!pip install -q openpyxl
import io
import pandas as pd
from google.colab import files

print("Choose sheet A first, then sheet B")
up_a = files.upload()
up_b = files.upload()
A = pd.read_excel(io.BytesIO(list(up_a.values())[0]), sheet_name="Annotate")
B = pd.read_excel(io.BytesIO(list(up_b.values())[0]), sheet_name="Annotate")

A = A[~A["ID"].astype(str).str.startswith("EXAMPLE")]
B = B[~B["ID"].astype(str).str.startswith("EXAMPLE")]

LABELS = {
    "Aspect 1 (exact words)": "aspect1", "Category 1": "category1", "Polarity 1": "polarity1",
    "Aspect 2 (exact words)": "aspect2", "Category 2": "category2", "Polarity 2": "polarity2",
    "Aspect 3 (exact words)": "aspect3", "Category 3": "category3", "Polarity 3": "polarity3",
    "Implied aspect category": "implied_category", "Implied aspect polarity": "implied_polarity",
    "Sarcastic?": "sarcastic", "Cultural reference?": "cultural_reference", "Language": "language",
}

a = A[["ID", "Domain"] + list(LABELS)].rename(columns={**LABELS, "ID": "review_id", "Domain": "domain"})
b = B[["ID"] + list(LABELS)].rename(columns={**LABELS, "ID": "review_id"})
combined = a.merge(b, on="review_id", suffixes=("_annotator1", "_annotator2"))

print("Reviews in A:", len(a), "| in B:", len(b), "| combined:", len(combined))
combined.to_csv("agreement_sample_200.csv", index=False, encoding="utf-8")
files.download("agreement_sample_200.csv")

Choose sheet A first, then sheet B


Saving AGREEMENT_Annotator_A_original_volunteer_labels.xlsx to AGREEMENT_Annotator_A_original_volunteer_labels (2).xlsx


Saving AGREEMENT_Annotator_B_completed.xlsx to AGREEMENT_Annotator_B_completed (2).xlsx
Reviews in A: 200 | in B: 200 | combined: 200


/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# I calculate Krippendorff's alpha step by step for polarity and sarcasm, and check it against the krippendorff package

!pip install -q krippendorff
import numpy as np
import pandas as pd
import krippendorff

df = pd.read_csv("agreement_sample_200.csv")

def aspects(row, who):
    found = {}
    for i in (1, 2, 3):
        word = row[f"aspect{i}_{who}"]
        if pd.notna(word):
            found[str(word).strip().lower()] = (row[f"category{i}_{who}"], row[f"polarity{i}_{who}"])
    if pd.notna(row[f"implied_category_{who}"]):
        found["(implied)"] = (row[f"implied_category_{who}"], row[f"implied_polarity_{who}"])
    return found

def explain(x, y, name):
    clean = lambda v: "blank" if pd.isna(v) else str(v)
    x, y = [clean(v) for v in x], [clean(v) for v in y]
    labels = sorted(set(x) | set(y))
    print(f"\n=== {name}: {len(x)} items, agreed on {sum(a == b for a, b in zip(x, y))}")
    print(pd.crosstab(pd.Series(x, name="annotator1"), pd.Series(y, name="annotator2")))
    table = pd.DataFrame(0.0, index=labels, columns=labels)
    for a, b in zip(x, y):
        table.loc[a, b] += 1
        table.loc[b, a] += 1
    total = table.values.sum()
    per_label = table.sum(axis=1)
    observed = (total - np.trace(table.values)) / total
    expected = (total ** 2 - (per_label ** 2).sum()) / (total * (total - 1))
    print(f"Observed disagreement Do = {observed:.4f} | Expected disagreement De = {expected:.4f}")
    print(f"alpha = 1 - Do/De = {1 - observed / expected:.3f}")
    code = {label: i for i, label in enumerate(labels)}
    data = np.array([[code[v] for v in x], [code[v] for v in y]], dtype=float)
    print("krippendorff package gives:", round(krippendorff.alpha(reliability_data=data, level_of_measurement="nominal"), 3))

pol1, pol2 = [], []
for _, row in df.iterrows():
    a, b = aspects(row, "annotator1"), aspects(row, "annotator2")
    for word in set(a) & set(b):
        pol1.append(a[word][1])
        pol2.append(b[word][1])

explain(pol1, pol2, "Polarity")
explain(df["sarcastic_annotator1"], df["sarcastic_annotator2"], "Sarcasm")


=== Polarity: 277 items, agreed on 272
annotator2  negative  neutral  positive
annotator1                             
negative         202        2         0
neutral            1        2         1
positive           0        1        68
Observed disagreement Do = 0.0181 | Expected disagreement De = 0.3987
alpha = 1 - Do/De = 0.955
krippendorff package gives: 0.955

=== Sarcasm: 200 items, agreed on 190
annotator2  blank   no  yes
annotator1                 
blank           3    0    0
no              3  182    4
yes             1    2    5
Observed disagreement Do = 0.0500 | Expected disagreement De = 0.1283
alpha = 1 - Do/De = 0.610
krippendorff package gives: 0.61
